In [ ]:
import os
from dotenv import load_dotenv
import requests
import pandas as pd
import json
import time
from datetime import datetime

In [ ]:
def load_api_key():
    """
    Load Steam API keys from the .env file located in the .venv folder.
    
    Returns:
        tuple: A tuple containing (api_key).
    """
    # Notebook is in the 'src/' folder, so go up one level to reach '.venv/.env'
    env_path = r'..\src\config.env'
    
    load_dotenv(dotenv_path=env_path)
    api_key = os.getenv('itd_api_key')
    return api_key

api_key = load_api_key()


In [ ]:
games = pd.read_json(r'..\data\games_id_all.json')
games = games.loc['apps', 'response']

def name_formatting(id):
    game_data = games[id]
    #print(games.loc['apps', 'response'][id])
    game_name = game_data['name']
    game_id = game_data['appid']
    formatted_name = game_name.lower().strip().replace(' ', '-').replace(':', '').replace("'", '')
    return formatted_name, game_id

In [ ]:
def date_formatting(date):
    date = date.replace(',', '').split(' ')
    
    match date[1]:
        case 'Jan':
            date[1] = '1'
        case 'Feb':
            date[1] = '2'
        case 'Mar':
            date[1] = '3'
        case 'Apr':
            date[1] = '4'
        case 'May':
            date[1] = '5'
        case 'Jun':
            date[1] = '6'
        case 'Jul':
            date[1] = '7'
        case 'Aug':
            date[1] = '8'
        case 'Sep':
            date[1] = '9'
        case 'Oct':
            date[1] = '10'
        case 'Nov':
            date[1] = '11'
        case 'Dec':
            date[1] = '12'
        case _:
            print('Error: Invalid month')

    new_date = date[0] + '-' + date[1] + '-' + date[2]

    date_format = '%d-%m-%Y'

    date_datetime = datetime.strptime(new_date, date_format)

    formatted_date = date_datetime.strftime('%Y-%m-%dT%H:%M:%S+01:00')

    return formatted_date

In [ ]:
def used_ids_write(id):
    with open(r"..\data\games_prices\used_ids.txt", "a", encoding="utf-8") as id_file:
        id_file.write(str(id) + "\n")

def jsonl_write(name, new_entry):
    with open(rf"..\data\games_prices\{name}", "a", encoding="utf-8") as jsonl_file:
        jsonl_file.write(json.dumps(new_entry) + "\n")

In [ ]:
chunk = 1
chunk_name = f'price_data_chunk_{chunk}.jsonl'
game_chunk = f'games_chunk_{chunk}.jsonl'

try:
    with open(r"..\data\games_prices\used_ids.txt", 'r', encoding='utf-8') as id_file:
        last_id = id_file[-1:]
except:
    last_id = -1

for index, game in enumerate(games):
    if(index <= last_id):
        continue

    if(index == 10000):          #temporary stop
        break
    if(index % 80 == 0):
        time.sleep(40)

    URL = 'https://api.isthereanydeal.com/games/search/v1'

    game_name, app_id = name_formatting(index)
    new_entry = {"appid": app_id, "name": game_name}

    with open(rf"..\data\games_informations\{game_chunk}", "r", encoding="utf-8") as jsonl_file:
        try:
            for line in jsonl_file:
                game_details = json.loads(line)
                
                try:
                    game_release_date = game_details[str(app_id)]['release_date']['date']
                    game_release_date = date_formatting(game_release_date)
                    break
                except:
                    pass
            if game_details[str(app_id)]['is_free'] == 'true':
                new_entry.update({'is_free': 'true'})
                jsonl_write(chunk_name, new_entry)
                used_ids_write(app_id)
                continue
        except:
            print('Error: File does not contain this id')
    
    params = {
        'key': api_key,
        'title': game_name         #fetching game name to get uuid
    }

    response = requests.get(URL, params=params)

    if response.status_code == 200:
        data = response.json()

        game_id = ''
        for i in data:
            if i['slug'] == game_name:
                game_id = i["id"]

        if(game_id == ''):
            new_entry.update({'data_error': 'true'})
            jsonl_write(chunk_name, new_entry)
            used_ids_write(app_id)
            continue

        URL_price = 'https://api.isthereanydeal.com/games/history/v2'       #fetching  price data

        params = {
            'key': api_key,
            'id': game_id,
            'shops': 61,
            'country': 'PL',
            'since': game_release_date
            }

        response = requests.get(URL_price, params=params)

        if response.status_code == 200:
            data = response.json()

            #print(data)

            try:
                currency = data[0]['deal']['price']['currency']    
            except IndexError:
                currency = 'N/A'
            new_entry.update({'currency': currency})

            for i in data:
                del i['shop']
                del i['deal']['regular'] 
                
                del i['deal']['price']['currency']               #necessary data
            
            new_entry.update({'price_changes': data})
            jsonl_write(chunk_name, new_entry)

            used_ids_write(app_id)
            
        else:
            print(f"Error during prices fetching: {response.status_code}")
    else:
        print(f"Error during id fetching: {response.status_code}")